In [12]:
import gradio as gr
import sqlite3
import hashlib
import os
import shutil

conn = sqlite3.connect("database.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE,
    password_hash TEXT
)
""")
conn.commit()

In [13]:
def hash_password(p):
    return hashlib.sha256(p.encode()).hexdigest()


In [14]:
def signup(username, password):
    try:
        cursor.execute(
            "INSERT INTO users (username, password_hash) VALUES (?, ?)",
            (username, hash_password(password))
        )
        conn.commit()
        return "✅ Account created"
    except:
        return "❌ Username exists"


def login(username, password):
    cursor.execute(
        "SELECT id FROM users WHERE username=? AND password_hash=?",
        (username, hash_password(password))
    )
    row = cursor.fetchone()

    if row:
        # RETURN dict ONLY to State output
        return {"user_id": row[0], "username": username}, "✅ Logged in"
    else:
        return None, "❌ Invalid login"


def logout():
    return None, "🔒 Logged out"


In [15]:
def save_note(note, user_state):
    if user_state is None:
        return "❌ Please login first"
    return f"✅ Saved note for user {user_state['username']}"


In [24]:
with gr.Blocks() as app:

    user_state = gr.State()  # ✅ ONLY defined here

    gr.Markdown("## 🔐 Login")

    username = gr.Textbox(label="Username")
    password = gr.Textbox(label="Password", type="password")
    
    login_btn = gr.Button("Login")
    signup_btn = gr.Button("Signup")
    logout_btn = gr.Button("Logout")

    status = gr.Textbox(label="Status")
    load_btn = gr.Button("Login")
    note = gr.Textbox(label="Note")
    save_btn = gr.Button("Save Note")

    # ---- AUTH ----
    login_btn.click(
        login,
        inputs=[username, password],
        outputs=[user_state, status]
    )
    load_btn.click(
        load_notes,
        inputs=user_state,
        outputs=notes_view
    )


    signup_btn.click(
        signup,
        inputs=[username, password],
        outputs=status
    )

    logout_btn.click(
        logout,
        outputs=[user_state, status]
    )

    # ---- PROTECTED ----
    save_btn.click(
        save_note,
        inputs=[note, user_state],   # ✅ state as INPUT
        outputs=status
    )

app.launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [17]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS notes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    content TEXT
)
""")
conn.commit()

In [22]:
def save_note(note, user_state):
    if user_state is None:
        return "❌ Please login first"

    cursor.execute(
        "INSERT INTO notes (user_id, content) VALUES (?, ?)",
        (user_state["user_id"], note)
    )
    conn.commit()

    return "✅ Note saved successfully ✅"


In [19]:
def load_notes(user_state):
    if user_state is None:
        return "❌ Please login first"

    cursor.execute(
        "SELECT content FROM notes WHERE user_id=?",
        (user_state["user_id"],)
    )
    rows = cursor.fetchall()

    if not rows:
        return "📭 No notes yet"

    return "\n\n---\n\n".join(r[0] for r in rows)


In [20]:
notes_view = gr.Textbox(
    label="📒 My Notes",
    lines=10,
    interactive=False
)

load_btn = gr.Button("Load My Notes")
